
# Grid Search for TreeLUT Hyperparameters

In [1]:
import os
import numpy as np
import pandas as pd
import scipy.io
import csv         # <--- MODIFIED
import threading   # <--- MODIFIED
import requests
from tqdm import tqdm
import time
import joblib  # Added for parallelism
import shutil
import re
import subprocess # Added for Yosys
import copy      # Added for parameter management
import json      # Added for parsing Yosys report
from datetime import datetime
import itertools # <-- The missing import

from xgboost import XGBClassifier
from treelut import TreeLUTClassifier
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.metrics import accuracy_score

# --- Imports ---
import json
import os
import copy
import subprocess
import shutil
import itertools
from datetime import datetime

import pandas as pd
import numpy as np
import joblib
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier  # <--- MODIFIED: Added LightGBM

import multiprocessing.resource_tracker
multiprocessing.resource_tracker._CLEANUP_FUNCS.pop('folder', None)


# --- Data Loading (from your notebook cell 011f2f3a) ---
def load_image(image_info, data_path="."):
    """Loads the image and the ground truth from a `mat` file."""
    image_name = image_info['key']
    if not os.path.exists(data_path):
        os.makedirs(data_path)
    
    input_file = os.path.join(data_path, image_info['file'])
    label_file = os.path.join(data_path, image_info['file_gt'])
    
    try:
        X = scipy.io.loadmat(input_file)[image_name]
    except:
        print(f"Downloading {image_info['file']}...")
        os.system(f"wget {image_info['url']} -O {input_file}")
        X = scipy.io.loadmat(input_file)[image_name]
    
    try:
        y = scipy.io.loadmat(label_file)[image_info['key_gt']]
    except:
        print(f"Downloading {image_info['file_gt']}...")
        os.system(f"wget {image_info['url_gt']} -O {label_file}")
        y = scipy.io.loadmat(label_file)[image_info['key_gt']]
    
    return X, y

def pixel_classification_preprocessing(X, y):
    """Preprocesses hyperspectral images for pixel classification."""
    X = X.reshape(-1, X.shape[2])
    y = y.reshape(-1)
    X = X[y > 0, :]
    y = y[y > 0]
    for new_class_num, old_class_num in enumerate(np.unique(y)):
        y[y == old_class_num] = new_class_num
    return X, y

# --- Script Loading (Updated) ---
def load_yosys_script_from_txt():
    """Reads the Yosys script content from its file."""
    try:
        # Assuming the script is in a 'misc' folder
        with open("misc/yosys_script.ys", "r", encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        print("Error: 'misc/yosys_script.ys' not found.")
        return None

In [2]:
# --- Imports ---
import json
import os
import copy
import subprocess
import shutil
import itertools
from datetime import datetime
import csv         # <--- MODIFIED
import threading   # <--- MODIFIED

import pandas as pd
import numpy as np
import joblib
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# (Assuming TreeLUTClassifier is imported)
# from treelut.classifier import TreeLUTClassifier 

# <--- MODIFIED: Define all possible CSV columns for consistency ---
CSV_FIELDNAMES = [
    'timestamp', 'dataset', 'model_type', 'n_features', 'n_classes',
    'n_trees', 'max_depth', 'eta', 'w_feature', 'w_tree', 'style',
    'argmax', 'quantization', 'n_nodes', 'lut', 'accuracy',
    'dir_path', 'error'
]

# --- Unchanged Function ---
def _parse_luts_from_yosys_json(report_path):
    """Parses the 'stat -json' output to get the total LUT count."""
    luts = 0
    try:
        with open(report_path, "r") as f:
            report = json.load(f)
        luts = report.get("design", {}).get("num_cells_by_type", {}).get("$lut", {})
        total_luts = sum(luts.values()) if isinstance(luts, dict) else luts
        return total_luts
    except json.JSONDecodeError:
        print(f"Warning: Could not decode JSON from {report_path}")
        return -1
    except FileNotFoundError:
        print(f"Warning: Synthesis report not found at {report_path}")
        return -1
    except Exception as e:
        print(f"Warning: Error parsing LUTs from {report_path}: {e}")
        return -1

# <--- MODIFIED: New thread-safe CSV writing function ---
def _save_result_to_csv(result_dict, csv_filename, fieldnames, lock):
    """
    Appends a single result dictionary to a CSV file in a thread-safe manner.
    """
    try:
        with lock: # Acquire the lock to prevent simultaneous writes
            # Check if file exists to determine if we need to write headers
            file_exists = os.path.exists(csv_filename)
            
            with open(csv_filename, 'a', newline='', encoding='utf-8') as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                
                if not file_exists:
                    writer.writeheader() # Write header only if file is new
                    
                writer.writerow(result_dict)
                
    except Exception as e:
        # This is a critical error if saving fails
        print(f"--- CRITICAL: FAILED TO SAVE RESULT TO CSV {csv_filename} ---")
        print(f"Error: {e}")
        print(f"Data that was not saved: {result_dict}")
# --- End of new function ---


# --- Modified Worker Function ---
def _run_single_experiment(params_tuple, xgb_params_base, lgbm_params_base,
                           treelut_params_base, 
                           yosys_script_content, 
                           datasets_map, datasets_name_map, results_parent_dir,
                           csv_filename, csv_fieldnames, lock): # <--- MODIFIED: Added args
    """
    Worker function for a single experiment.
    Saves its own result to CSV immediately using a thread-safe lock.
    """
    # --- 1. Unpack Params & Set Up Paths ---
    (dataset_index, dataset_key), w_feature, w_tree, quant, arg, s, model_type, n_estimators, max_d, eta = params_tuple
    
    dataset_name = datasets_name_map[dataset_index]
    X, y = datasets_map[dataset_key]
    y = pd.Series(y) 
    
    name = (f"{dataset_name}_model_{model_type}_est_{n_estimators}_eta_{eta}_maxd_{max_d}_"
            f"wf_{w_feature}_wt_{w_tree}_q_{quant}_arg_{arg}_s_{s}")
    
    dir_path = os.path.join(results_parent_dir, name)
    os.makedirs(dir_path, exist_ok=True)
    
    treelut_verilog_dir = os.path.join(dir_path, "TreeLUT", "verilog")
    os.makedirs(treelut_verilog_dir, exist_ok=True)
    
    result_dict = {} # <--- MODIFIED: Define dict up here

    try:
        # --- 2. Set Up Model Parameters ---
        # ... (Parameter setup code is unchanged) ...
        current_treelut_params = copy.deepcopy(treelut_params_base)
        current_treelut_params.update({
            'style': s, 'w_feature': w_feature, 'w_tree': w_tree,
            'dir_path': dir_path, 'quantized': quant, 'argmax': arg
        })

        n_classes = len(np.unique(y))
        model = None
        current_model_params = {}

        if model_type == 'xgb':
            current_model_params = copy.deepcopy(xgb_params_base)
            current_model_params.update({
                'num_class': n_classes, 'n_estimators': n_estimators,
                'max_depth': max_d, 'eta': eta
            })
            model = XGBClassifier(**current_model_params)
        elif model_type == 'lgbm':
            current_model_params = copy.deepcopy(lgbm_params_base)
            current_model_params.update({
                'objective': 'multiclass', 'num_class': n_classes,
                'n_estimators': n_estimators, 'max_depth': max_d,
                'learning_rate': eta
            })
            model = LGBMClassifier(**current_model_params)
        else:
            raise ValueError(f"Unknown model_type: {model_type}")

        # --- 3. Split & Quantize Data ---
        # ... (Data split/quantize code is unchanged) ...
        X_train_raw, X_test_raw, y_train, y_test = train_test_split(
            X, y, test_size=0.3, random_state=42, stratify=y
        )
        scaler = MinMaxScaler()
        X_train_scaled = scaler.fit_transform(X_train_raw)
        X_test_scaled = scaler.transform(X_test_raw)
        min_vals, max_vals = scaler.data_min_, scaler.data_max_
        quant_val = (2**w_feature - 1)
        X_train_q = np.round(X_train_scaled * quant_val)
        X_test_q = np.clip(np.round(X_test_scaled * quant_val), 0, quant_val)

        # --- 4. Train Model ---
        model.fit(X_train_q, y_train)

        # --- 5. Run TreeLUT (Convert, Predict, Verilog) ---
        # ... (TreeLUT instantiation logic is unchanged) ...
        treelut_clf = None
        treelut_init_args = {
            **current_treelut_params, 'min': min_vals, 'max': max_vals
        }
        if model_type == 'xgb':
            treelut_clf = TreeLUTClassifier(xgb_model=model, **treelut_init_args)
        elif model_type == 'lgbm':
            treelut_clf = TreeLUTClassifier(xgb_model=model, **treelut_init_args)
        if treelut_clf is None:
            raise ValueError(f"TreeLUTClassifier setup failed for model_type: {model_type}")

        treelut_clf.convert()
        y_pred = treelut_clf.predict(X_test_q)
        accuracy = accuracy_score(y_test, y_pred)
        
        try:
            nodes = sum(treelut_clf.nodes())
        except AttributeError:
            nodes = -1

        treelut_clf.verilog()
        treelut_clf.testbench(X_test_q, y_test) 
        
        # --- 6. Run Yosys Synthesis ---
        # ... (Yosys script/run code is unchanged) ...
        yosys_script_path = os.path.join(dir_path, "yosys_script.ys")
        report_json_path = os.path.join(dir_path, "synthesis_report.json")
        report_log_path = os.path.join(dir_path, "yosys_log.txt")
        with open(yosys_script_path, "w", encoding="utf-8") as f:
            f.write(yosys_script_content.format(dir_path=dir_path))
        with open(report_log_path, "w") as f_log:
            subprocess.run(["yosys", "-Q", "-s", yosys_script_path], 
                           stdout=f_log, stderr=subprocess.STDOUT)

        # --- 7. Parse LUTs from Yosys's JSON Report ---
        luts = _parse_luts_from_yosys_json(report_json_path)

        # --- 8. Organize Artifacts (Optional) ---
        # ... (Artifact move code is unchanged) ...
        result_blif_path = os.path.join(dir_path, "result.blif")
        if os.path.exists(result_blif_path):
            new_blif_name = f"{name}.blif"
            shutil.move(result_blif_path, os.path.join(results_parent_dir, new_blif_name))

        # --- 9. Prepare Result Dictionary ---
        # <--- MODIFIED: Populate dict, add 'error' key for consistency ---
        result_dict = {
            'timestamp': pd.Timestamp.now(),
            'dataset': dataset_name,
            'model_type': model_type,
            'n_features': X.shape[1],
            'n_classes': len(np.unique(y)),
            'n_trees': n_estimators,
            'max_depth': max_d,
            'eta': eta,
            'w_feature': w_feature,
            'w_tree': w_tree,
            'style': s,
            'argmax': arg,
            'quantization': quant,
            'n_nodes': nodes,
            'lut': luts,
            'accuracy': accuracy,
            'dir_path': dir_path,
            'error': np.nan # Add 'error' key with null value
        }
    
    except Exception as e:
        print(f"--- FAILED RUN: {name} ---")
        print(f"Error: {e}")
        # <--- MODIFIED: Populate dict, add missing keys for consistency ---
        result_dict = {
            'timestamp': pd.Timestamp.now(),
            'dataset': dataset_name,
            'model_type': model_type,
            'n_features': np.nan, # Add missing key
            'n_classes': np.nan, # Add missing key
            'n_trees': n_estimators,
            'max_depth': max_d,
            'eta': eta,
            'w_feature': w_feature,
            'w_tree': w_tree,
            'style': s,
            'argmax': arg,
            'quantization': quant,
            'n_nodes': -1,
            'lut': -1,
            'accuracy': np.nan,
            'dir_path': dir_path,
            'error': str(e)
        }
    
    # <--- MODIFIED: Save result to CSV before returning ---
    _save_result_to_csv(result_dict, csv_filename, csv_fieldnames, lock)
    
    return result_dict
    

# --- Modified Main Function ---
def treelut_grid_search_with_synthesis(datasets_map, datasets_name_map, 
                                     xgb_params_base, lgbm_params_base, 
                                     treelut_params_base, 
                                     param_grid, results_parent_dir="results", n_jobs=-1):
    """
    Main function for parallel grid search.
    Results are saved incrementally to a CSV file.
    """
    
    print("--- 1. Loading Yosys Script ---")
    yosys_script_content = load_yosys_script_from_txt()
    if not yosys_script_content:
        print("Error: Yosys script not found. Aborting.")
        return None
    
    if 'model_type' not in param_grid:
        print("Error: 'model_type' (e.g., ['xgb', 'lgbm']) must be in param_grid. Aborting.")
        return None

    # <--- MODIFIED: Set up CSV file and lock ---
    csv_filename = f"treelut_synthesis_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    lock = threading.Lock()
    print(f"Results will be saved incrementally to {csv_filename}")
    # --- End of MODIFIED block ---

    # --- 2. Create All Parameter Combinations ---
    all_combinations = list(itertools.product(
        list(enumerate(datasets_map.keys())),
        param_grid['w_feature'],
        param_grid['w_tree'],
        param_grid['quantized'],
        param_grid['argmax'],
        param_grid['style'],
        param_grid['model_type'],
        param_grid['n_estimators'],
        param_grid['max_depth'],
        param_grid['eta']
    ))
    
    print(f"--- 2. Starting Parallel Grid Search ---")
    print(f"Total combinations to test: {len(all_combinations)}")
    
    # --- 3. Run in Parallel ---
    results_list = joblib.Parallel(
        n_jobs=n_jobs,
        backend="threading"
    )(
        joblib.delayed(_run_single_experiment)(
            params,
            xgb_params_base,
            lgbm_params_base,
            treelut_params_base,
            yosys_script_content,
            datasets_map,
            datasets_name_map,
            results_parent_dir,
            csv_filename,    # <--- MODIFIED
            CSV_FIELDNAMES,  # <--- MODIFIED
            lock             # <--- MODIFIED
        ) for params in tqdm(all_combinations, desc="Grid Search")
    )

    # --- 4. Collect and Save Results ---
    print(f"\n--- 3. Grid Search Complete. Results are saved in {csv_filename} ---")
    
    # Filter out potential None results if any (though _save_... handles this)
    results_list = [r for r in results_list if r is not None]
    
    # We can still create and return the full DataFrame for inspection
    results_df = pd.DataFrame(results_list)
    
    # <--- MODIFIED: Removed the final .to_csv() call as it's now redundant ---
    
    if results_df.empty:
        print("Warning: No results were collected.")
        return pd.DataFrame()

    return results_df.sort_values('accuracy', ascending=False)




In [3]:
# --- 1. Load All Datasets into a Dictionary ---
# (from your notebook cell 4ba5d3e4)
IMAGES = {
    # "indian_pines": {
    #     "file": "Indian_pines_corrected.mat",
    #     "file_gt": "Indian_pines_gt.mat",
    #     "key": "indian_pines_corrected",
    #     "key_gt": "indian_pines_gt",
    #     "url": "http://www.ehu.eus/ccwintco/uploads/6/67/Indian_pines_corrected.mat",
    #     "url_gt": "http://www.ehu.eus/ccwintco/uploads/c/c4/Indian_pines_gt.mat",
    # },

    # "KSC": {
    #     "file": "KSC.mat",
    #     "file_gt": "KSC_gt.mat",
    #     "key": "KSC",
    #     "key_gt": "KSC_gt",
    #     "url": "http://www.ehu.es/ccwintco/uploads/2/26/KSC.mat",
    #     "url_gt": "http://www.ehu.es/ccwintco/uploads/a/a6/KSC_gt.mat",
    #     "p": 0.15,
    #     'class_names': {
    #             0: 'Unlabeled',
    #             1: 'Scrub',
    #             2: 'Willow swamp',
    #             3: 'CP hammock',
    #             4: 'Slash pine',
    #             5: 'Oak/Broadleaf',
    #             6: 'Hardwood',
    #             7: 'Swamp',
    #             8: 'Graminoid marsh',
    #             9: 'Spartina marsh',
    #             10: 'Cattail marsh',
    #             11: 'Salt marsh',
    #             12: 'Mud flats',
    #             13: 'Water'}
    # },
    
    "paviaU": {
        "file": "PaviaU.mat",
        "file_gt": "PaviaU_gt.mat",
        "key": "paviaU",
        "key_gt": "paviaU_gt",
        "url": "http://www.ehu.eus/ccwintco/uploads/e/ee/PaviaU.mat",
        "url_gt": "http://www.ehu.eus/ccwintco/uploads/5/50/PaviaU_gt.mat",
        "p": 0.15
    },
    
    # "salinas": {
    #     "file": "Salinas.mat",
    #     "file_gt": "Salinas_gt.mat",
    #     "key": "salinas",
    #     "key_gt": "salinas_gt",
    #     "url": "http://www.ehu.eus/ccwintco/uploads/f/f1/Salinas.mat",
    #     "url_gt": "http://www.ehu.eus/ccwintco/uploads/f/fa/Salinas_gt.mat",
    #     "p": 0.15
    # }
}

# Pre-load and preprocess all datasets
datasets_map = {}
datasets_name_map = []
for i, (name, info) in enumerate(IMAGES.items()):
    print(f"Loading dataset: {name}")
    data, labels = load_image(info, data_path="./data")
    X_processed, y_processed = pixel_classification_preprocessing(data, labels)
    datasets_map[name] = (X_processed, y_processed)
    datasets_name_map.append(name)


# --- 2. Define Base Parameters (from your notebook cell 00a3093f) ---
xgb_params_base = {
    'objective': 'multi:softmax',
    'random_state': 42,
    'verbosity': 0,
    'n_jobs': 1, # Use 1 for XGB to avoid conflict with joblib parallel runs
    'device': 'cuda' 
}

lgbm_params_base = {
    'objective': 'multiclass',
    'random_state': 42,
    'verbosity': -1,
}


treelut_params_base = {
    'bits_features': 16, # From your notebook
    'pipeline': [0, 0, 0], # From your notebook
}

# --- 3. Define Parameter Grid (from your script) ---
# This is the grid you provided
param_grid = {
    'model_type': ['lgbm'],
    'w_feature': [ 8],
    'w_tree': [ 8],
    'quantized': [False, True],
    'argmax': [False, True],
    'style': ['mux', 'equation'],
    'n_estimators': [13],
    'max_depth': [6],
    'eta': [0.1]
}

# --- 4. Run the Parallel Grid Search ---
results_parent_dir = "results_grid_search" # Master folder for all runs
os.makedirs(results_parent_dir, exist_ok=True)

results_df = treelut_grid_search_with_synthesis(
    datasets_map=datasets_map,
    datasets_name_map=datasets_name_map,
    xgb_params_base=xgb_params_base,
    lgbm_params_base=lgbm_params_base,
    treelut_params_base=treelut_params_base,
    param_grid=param_grid,
    results_parent_dir=results_parent_dir,
    n_jobs=11 # Use all available CPU cores
)



Loading dataset: paviaU
--- 1. Loading Yosys Script ---
Results will be saved incrementally to treelut_synthesis_results_20251113_140801.csv
--- 2. Starting Parallel Grid Search ---
Total combinations to test: 8


Grid Search: 100%|██████████| 8/8 [00:00<00:00, 44858.87it/s]



--- 3. Grid Search Complete. Results are saved in treelut_synthesis_results_20251113_140801.csv ---


In [4]:
# --- 5. Display Top 5 Results ---
print("\n--- Top 5 Results (Accuracy & LUTs) ---")
results_df[['dataset', 'accuracy', 'lut', 'n_nodes', 'n_trees', 'max_depth', 'w_feature', 'w_tree', 'quantization', 'argmax', 'style']].head(5)


--- Top 5 Results (Accuracy & LUTs) ---


,dataset,accuracy,lut,n_nodes,n_trees,max_depth,w_feature,w_tree,quantization,argmax,style
0,paviaU,0.885763,5995,6741,13,6,8,8,False,False,mux
1,paviaU,0.885763,4890,6741,13,6,8,8,False,False,equation
2,paviaU,0.885763,6613,6741,13,6,8,8,False,True,mux
3,paviaU,0.885763,5323,6741,13,6,8,8,False,True,equation
4,paviaU,0.885763,-1,6741,13,6,8,8,True,False,mux
